# 05 — Configuration Benchmark + LangSmith — Simple Decision Version

This version is deliberately simpler.

We make only **three decisions**:

1. **Chunk size:** 1200 vs 600
2. **Retrieval k:** 3 vs 5
3. **Reranker:** OFF vs ON

For each decision:

- run the **same fixed questions**,
- show the retrieved evidence clearly in the notebook,
- show a small summary,
- make one decision,
- then move to the next test.

## Baseline

```text
chunk = 1200
k = 3
reranker = OFF
```

## LangSmith goal

Every benchmark question creates **one top-level LangSmith trace** named:

```text
benchmark_question
```

The trace input includes the full `question` text first, so the question is easy to see in LangSmith.

The trace output includes the retrieved evidence, lecture numbers, timestamps, and latency.


# Step 0 — Setup

This keeps the same paths and models as the working multi-lecture RAG notebook.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
!pip install -q \
    langchain \
    langchain-core \
    langchain-chroma \
    langchain-huggingface \
    sentence-transformers \
    langsmith \
    pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
from pathlib import Path
import json
import os
import time
import shutil
import pandas as pd

from google.colab import userdata
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import CrossEncoder
import langsmith as ls
from langsmith import traceable

PROJECT_ROOT = Path("/content/drive/MyDrive/AI_Engineering_Final_Project")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TEXT_DIR = PROCESSED_DIR / "transcripts"
METADATA_DIR = PROCESSED_DIR / "metadata"

RAG_DIR = PROJECT_ROOT / "rag"
BENCHMARK_DIR = RAG_DIR / "benchmark_l01_l05"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LECTURE_NUMBERS = [1, 2, 3, 4, 5]

BASELINE_CHUNK = 1200
ALT_CHUNK = 600
BASELINE_K = 3
ALT_K = 5
RERANK_TOP_N = 3

print("Transcripts:", TEXT_DIR)
print("Metadata:", METADATA_DIR)
print("Benchmark directory:", BENCHMARK_DIR)


# Step 1 — LangSmith setup

This uses the same working pattern as Notebook 4:

- EU endpoint
- workspace ID
- existing project `students-channel-brain`
- explicit `langsmith_client`

LangSmith is required for this notebook because the trainer wants logs and comments on performance impact.


In [ ]:
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")
LANGSMITH_WORKSPACE_ID = userdata.get("LANGSMITH_WORKSPACE_ID")

assert LANGSMITH_API_KEY, "LANGSMITH_API_KEY missing from Colab Secrets."
assert LANGSMITH_WORKSPACE_ID, "LANGSMITH_WORKSPACE_ID missing from Colab Secrets."

langsmith_client = ls.Client(
    api_key=LANGSMITH_API_KEY,
    api_url="https://eu.api.smith.langchain.com",
    workspace_id=LANGSMITH_WORKSPACE_ID,
)

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_WORKSPACE_ID"] = LANGSMITH_WORKSPACE_ID
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "students-channel-brain"

print("LangSmith ready")
print("Project: students-channel-brain")
print("Endpoint:", os.environ["LANGSMITH_ENDPOINT"])


LangSmith ready
Project: students-channel-brain
Endpoint: https://eu.api.smith.langchain.com


# Step 2 — Validate and load Lectures 1–5

We stop immediately if any transcript or metadata file is missing.


In [ ]:
lecture_records = []

for lecture_number in LECTURE_NUMBERS:
    lecture_id = f"lecture_{lecture_number:02d}"

    stt_path = TEXT_DIR / f"{lecture_id}_stt.json"
    metadata_path = METADATA_DIR / f"{lecture_id}_metadata.json"

    assert stt_path.exists(), f"Missing: {stt_path}"
    assert metadata_path.exists(), f"Missing: {metadata_path}"

    with open(stt_path, "r", encoding="utf-8") as f:
        stt_segments = json.load(f)

    with open(metadata_path, "r", encoding="utf-8") as f:
        lecture_metadata = json.load(f)

    lecture_records.append({
        "lecture_number": lecture_number,
        "lecture_id": lecture_id,
        "stt_segments": stt_segments,
        "metadata": lecture_metadata,
    })

    print(
        f"Lecture {lecture_number}: "
        f"{len(stt_segments)} STT segments"
    )

print("\nLoaded", len(lecture_records), "lectures.")


Lecture 1: 280 STT segments
Lecture 2: 733 STT segments
Lecture 3: 398 STT segments
Lecture 4: 233 STT segments
Lecture 5: 682 STT segments

Loaded 5 lectures.


# Step 3 — Timestamp-aware chunking

This is the same chunking style used in the working RAG notebook.

For the chunk benchmark, only `max_chars` changes:

```text
600 vs 1200
```

`overlap_segments=1` stays fixed.


In [ ]:
def seconds_to_timestamp(seconds):
    seconds = int(seconds)
    minutes = seconds // 60
    secs = seconds % 60
    return f"{minutes:02d}:{secs:02d}"


def create_timestamp_chunks(
    segments,
    max_chars=1200,
    overlap_segments=1,
):
    chunks = []
    current = []
    current_chars = 0

    for segment in segments:
        text = segment["text"].strip()

        if not text:
            continue

        if current and current_chars + len(text) > max_chars:
            chunks.append({
                "text": " ".join(s["text"].strip() for s in current),
                "start_seconds": current[0]["start_seconds"],
                "end_seconds": current[-1]["end_seconds"],
            })

            current = current[-overlap_segments:] if overlap_segments else []
            current_chars = sum(len(s["text"]) for s in current)

        current.append(segment)
        current_chars += len(text)

    if current:
        chunks.append({
            "text": " ".join(s["text"].strip() for s in current),
            "start_seconds": current[0]["start_seconds"],
            "end_seconds": current[-1]["end_seconds"],
        })

    return chunks


def build_documents(chunk_size):
    documents = []

    for record in lecture_records:
        chunks = create_timestamp_chunks(
            record["stt_segments"],
            max_chars=chunk_size,
            overlap_segments=1,
        )

        meta = record["metadata"]

        for i, chunk in enumerate(chunks):
            documents.append(
                Document(
                    page_content=chunk["text"],
                    metadata={
                        "chunk_id": f"L{record['lecture_number']:02d}_C{i:03d}",
                        "lecture_id": record["lecture_id"],
                        "lecture_number": record["lecture_number"],
                        "lecture_title": meta.get(
                            "lecture_title",
                            f"Lecture {record['lecture_number']}",
                        ),
                        "start_seconds": float(chunk["start_seconds"]),
                        "end_seconds": float(chunk["end_seconds"]),
                        "video_url": meta.get("video_url", ""),
                        "benchmark_chunk_size": chunk_size,
                    },
                )
            )

    return documents


documents_600 = build_documents(600)
documents_1200 = build_documents(1200)

print("600-char chunks:", len(documents_600))
print("1200-char chunks:", len(documents_1200))


600-char chunks: 294
1200-char chunks: 130


# Step 4 — Embedding model + benchmark vector stores

The embedding model is fixed. It is **not** part of the benchmark.


In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

REBUILD_BENCHMARK_INDEX = True


def build_store(documents, chunk_size):
    persist_dir = BENCHMARK_DIR / f"chroma_chunk_{chunk_size}"
    collection_name = f"benchmark_l01_l05_chunk_{chunk_size}"

    if REBUILD_BENCHMARK_INDEX and persist_dir.exists():
        shutil.rmtree(persist_dir)

    store = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_model,
        persist_directory=str(persist_dir),
    )

    if len(store.get()["ids"]) == 0:
        store.add_documents(documents)

    print(
        f"{chunk_size} index ready | "
        f"{len(store.get()['ids'])} chunks"
    )

    return store


vectorstore_600 = build_store(documents_600, 600)
vectorstore_1200 = build_store(documents_1200, 1200)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

600 index ready | 294 chunks
1200 index ready | 130 chunks


# Step 5 — Fixed Tutor test set

This is the **full set of questions we want to test with the Tutor**.

It contains three groups:

1. **Retrieval/configuration questions** — used to compare chunk size, `k`, and reranker.
2. **Memory sequence** — must be asked in the same Tutor session/thread.
3. **Out-of-scope questions** — used to verify the Tutor refuses unsupported topics.

## Important

Only the `retrieval` questions are used for the chunk / k / reranker benchmark.

The `memory` and `out_of_scope` questions are still part of the Tutor evaluation set, but they test the **application/agent behavior**, not retrieval configuration.


In [ ]:
ALL_TUTOR_TESTS = [
    # ---------------------------------------------------------
    # A) Retrieval/configuration questions
    # ---------------------------------------------------------
    {
        "id": "R1",
        "category": "retrieval",
        "question": "What is the row picture of a system of linear equations?",
        "expected_lecture": 1,
    },
    {
        "id": "R2",
        "category": "retrieval",
        "question": "What is the column picture?",
        "expected_lecture": 1,
    },
    {
        "id": "R3",
        "category": "retrieval",
        "question": "What is a linear combination in the column picture?",
        "expected_lecture": 1,
    },
    {
        "id": "R4",
        "category": "retrieval",
        "question": "What is elimination and why do we use it?",
        "expected_lecture": 2,
    },
    {
        "id": "R5",
        "category": "retrieval",
        "question": "What is discussed about inverse matrices in Lecture 3?",
        "expected_lecture": 3,
    },
    {
        "id": "R6",
        "category": "retrieval",
        "question": "In Lecture 4, why does the factorization A = LU matter?",
        "expected_lecture": 4,
    },
    {
        "id": "R7",
        "category": "retrieval",
        "question": "How is LU related to elimination?",
        "expected_lecture": 4,
    },
    {
        "id": "R8",
        "category": "retrieval",
        "question": "What is the difference between the row picture and the column picture?",
        "expected_lecture": 1,
    },

    # ---------------------------------------------------------
    # B) Memory sequence — ask in this exact order,
    #    in the SAME Tutor session/thread
    # ---------------------------------------------------------
    {
        "id": "M1",
        "category": "memory",
        "question": "What is the column picture?",
        "expected_lecture": 1,
        "sequence": "memory_1",
        "turn": 1,
    },
    {
        "id": "M2",
        "category": "memory",
        "question": "What is the row picture?",
        "expected_lecture": 1,
        "sequence": "memory_1",
        "turn": 2,
    },
    {
        "id": "M3",
        "category": "memory",
        "question": "What are the differences between the two?",
        "expected_lecture": 1,
        "sequence": "memory_1",
        "turn": 3,
    },

    # ---------------------------------------------------------
    # C) Out-of-scope questions
    # ---------------------------------------------------------
    {
        "id": "O1",
        "category": "out_of_scope",
        "question": "Who won the World Cup?",
        "expected_lecture": None,
    },
    {
        "id": "O2",
        "category": "out_of_scope",
        "question": "Explain photosynthesis.",
        "expected_lecture": None,
    },
]

# Only these are used to compare chunk / k / reranker.
QUESTIONS = [
    item
    for item in ALL_TUTOR_TESTS
    if item["category"] == "retrieval"
]

MEMORY_TESTS = [
    item
    for item in ALL_TUTOR_TESTS
    if item["category"] == "memory"
]

OUT_OF_SCOPE_TESTS = [
    item
    for item in ALL_TUTOR_TESTS
    if item["category"] == "out_of_scope"
]

print("Full Tutor test set:", len(ALL_TUTOR_TESTS))
print("Retrieval benchmark questions:", len(QUESTIONS))
print("Memory questions:", len(MEMORY_TESTS))
print("Out-of-scope questions:", len(OUT_OF_SCOPE_TESTS))

display(
    pd.DataFrame(ALL_TUTOR_TESTS)[
        ["id", "category", "question", "expected_lecture"]
    ]
)


Full Tutor test set: 13
Retrieval benchmark questions: 8
Memory questions: 3
Out-of-scope questions: 2


,id,category,question,expected_lecture
0,R1,retrieval,What is the row picture of a system of linear ...,1.0
1,R2,retrieval,What is the column picture?,1.0
2,R3,retrieval,What is a linear combination in the column pic...,1.0
3,R4,retrieval,What is elimination and why do we use it?,2.0
4,R5,retrieval,What is discussed about inverse matrices in Le...,3.0
5,R6,retrieval,"In Lecture 4, why does the factorization A = L...",4.0
6,R7,retrieval,How is LU related to elimination?,4.0
7,R8,retrieval,What is the difference between the row picture...,1.0
8,M1,memory,What is the column picture?,1.0
9,M2,memory,What is the row picture?,1.0


# Step 6 — Reranker + one simple benchmark function

## Important simplification

There are no separate LangSmith traces for vector retrieval and reranking.

Instead, **one question = one LangSmith trace**.

That trace contains:

### Input
- full question
- case ID
- configuration name
- chunk size
- k
- reranker ON/OFF
- expected lecture

### Output
- top lecture
- latency
- retrieved lectures
- timestamps
- top evidence text

This makes the LangSmith log much easier to read.


In [ ]:
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"

reranker_model = CrossEncoder(RERANKER_MODEL)


def get_store(chunk_size):
    if chunk_size == 600:
        return vectorstore_600
    if chunk_size == 1200:
        return vectorstore_1200
    raise ValueError(f"Unsupported chunk size: {chunk_size}")


@traceable(name="benchmark_question")
def run_benchmark_question(
    question,
    case_id,
    config_name,
    chunk_size,
    k,
    reranker_on,
    expected_lecture,
):
    store = get_store(chunk_size)

    start = time.perf_counter()

    docs = store.similarity_search(
        question,
        k=k,
    )

    reranker_latency = 0.0

    if reranker_on:
        rerank_start = time.perf_counter()

        pairs = [
            [question, doc.page_content]
            for doc in docs
        ]

        scores = reranker_model.predict(pairs)

        docs = [
            doc
            for doc, score in sorted(
                zip(docs, scores),
                key=lambda item: float(item[1]),
                reverse=True,
            )
        ][:RERANK_TOP_N]

        reranker_latency = (
            time.perf_counter()
            - rerank_start
        )

    total_latency = time.perf_counter() - start

    retrieved_lectures = [
        int(doc.metadata["lecture_number"])
        for doc in docs
    ]

    top_lecture = (
        retrieved_lectures[0]
        if retrieved_lectures
        else None
    )

    evidence = [
        doc.page_content[:600].replace("\n", " ")
        for doc in docs
    ]

    result = {
        "question": question,
        "case_id": case_id,
        "config": config_name,
        "chunk_size": chunk_size,
        "k": k,
        "reranker": reranker_on,
        "expected_lecture": expected_lecture,
        "top_lecture": top_lecture,
        "top_lecture_correct": (
            top_lecture == expected_lecture
        ),
        "latency_s": round(total_latency, 4),
        "reranker_latency_s": round(reranker_latency, 4),
        "retrieved_lectures": retrieved_lectures,
        "timestamps": [
            seconds_to_timestamp(
                doc.metadata["start_seconds"]
            )
            for doc in docs
        ],
        "evidence": evidence,
    }

    return result


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

# Step 7 — Helper to run one comparison

This helper runs two configurations against the same questions and returns a simple table.

The notebook will call it three times:

1. chunk decision
2. k decision
3. reranker decision


In [ ]:
def run_configuration(config_name, config):
    rows = []

    for case in QUESTIONS:
        with ls.tracing_context(
            client=langsmith_client,
            project_name="students-channel-brain",
            enabled=True,
        ):
            result = run_benchmark_question(
                question=case["question"],
                case_id=case["id"],
                config_name=config_name,
                chunk_size=config["chunk_size"],
                k=config["k"],
                reranker_on=config["reranker"],
                expected_lecture=case["expected_lecture"],
            )

        rows.append(result)

    return pd.DataFrame(rows)


def make_side_by_side(left_df, right_df, left_name, right_name):
    left = left_df[
        [
            "case_id",
            "question",
            "top_lecture",
            "top_lecture_correct",
            "latency_s",
            "reranker_latency_s",
            "evidence",
        ]
    ].copy()

    right = right_df[
        [
            "case_id",
            "question",
            "top_lecture",
            "top_lecture_correct",
            "latency_s",
            "reranker_latency_s",
            "evidence",
        ]
    ].copy()

    left = left.rename(columns={
        "top_lecture": f"{left_name}_top_lecture",
        "top_lecture_correct": f"{left_name}_lecture_ok",
        "latency_s": f"{left_name}_latency",
        "reranker_latency_s": f"{left_name}_rerank_latency",
        "evidence": f"{left_name}_evidence",
    })

    right = right.drop(columns=["question"]).rename(columns={
        "top_lecture": f"{right_name}_top_lecture",
        "top_lecture_correct": f"{right_name}_lecture_ok",
        "latency_s": f"{right_name}_latency",
        "reranker_latency_s": f"{right_name}_rerank_latency",
        "evidence": f"{right_name}_evidence",
    })

    return left.merge(
        right,
        on="case_id",
        how="inner",
    )


def summarize(df, label):
    return {
        "configuration": label,
        "avg_latency_s": round(df["latency_s"].mean(), 4),
        "correct_top_lecture_rate": round(df["top_lecture_correct"].mean(), 3),
        "avg_reranker_latency_s": round(df["reranker_latency_s"].mean(), 4),
    }


# DECISION 1 — Chunk size: 1200 vs 600

Everything else is fixed:

```text
k = 3
reranker = OFF
```

## What to look at

For each question compare:

- Is the evidence actually about the question?
- Is the explanation complete enough?
- Is one chunk size too fragmented?
- Is one noticeably slower?

Do **not** decide only from the lecture number.


In [ ]:
chunk_1200 = run_configuration(
    "chunk_1200",
    {
        "chunk_size": 1200,
        "k": 3,
        "reranker": False,
    },
)

chunk_600 = run_configuration(
    "chunk_600",
    {
        "chunk_size": 600,
        "k": 3,
        "reranker": False,
    },
)

chunk_comparison = make_side_by_side(
    chunk_1200,
    chunk_600,
    "1200",
    "600",
)

display(chunk_comparison)

display(pd.DataFrame([
    summarize(chunk_1200, "1200"),
    summarize(chunk_600, "600"),
]))


,case_id,question,1200_top_lecture,1200_lecture_ok,1200_latency,1200_rerank_latency,1200_evidence,600_top_lecture,600_lecture_ok,600_latency,600_rerank_latency,600_evidence
0,R1,What is the row picture of a system of linear ...,1,True,0.0140,0.0,"[Let's just check this. If x is one, I have a ...",1,True,0.0102,0.0,"[This is my plan, the fundamental problem of l..."
1,R2,What is the column picture?,1,True,0.0102,0.0,[this one in the right amounts to get that one...,3,False,0.0099,0.0,"[here's a, so I call that column one. And what..."
2,R3,What is a linear combination in the column pic...,1,True,0.0099,0.0,[this one in the right amounts to get that one...,1,True,0.0098,0.0,"[You see the columns of the matrix, the column..."
3,R4,What is elimination and why do we use it?,2,True,0.0160,0.0,[Okay. This is it. This is then the second lec...,4,False,0.0099,0.0,[say some matrix A. Okay. Let's imagine it's 3...
4,R5,What is discussed about inverse matrices in Le...,4,False,0.0137,0.0,[transposes come in the opposite order. So it'...,4,False,0.0097,0.0,[on the other side. I have b inverse a inverse...
5,R6,"In Lecture 4, why does the factorization A = L...",4,True,0.0108,0.0,[is the most basic factorization of a matrix. ...,4,True,0.0098,0.0,[that we can now use. So now I put it to use. ...
6,R7,How is LU related to elimination?,2,False,0.0104,0.0,[It's the matrix that we needed to get this tw...,4,True,0.0098,0.0,"[elimination that comes from matrix, doing it ..."
7,R8,What is the difference between the row picture...,3,False,0.0111,0.0,[different from taking a row times a column. S...,3,False,0.0097,0.0,[chance to write down special matrices like th...


,configuration,avg_latency_s,correct_top_lecture_rate,avg_reranker_latency_s
0,1200,0.0120,0.625,0.0
1,600,0.0099,0.500,0.0


## Decision point 1


In [ ]:
CHUNK_DECISION = 1200

CHUNK_REASON = """
1200-character chunks performed better overall.

The correct top-lecture rate was 0.625 for chunk size 1200,
compared with 0.500 for chunk size 600.

The latency difference was very small:
about 0.0120 s for 1200 versus 0.0099 s for 600.

For several questions, 1200 also returned evidence from the expected lecture
where 600 did not. For example:
- R2: 1200 retrieved Lecture 1 correctly, while 600 retrieved Lecture 3.
- R4: 1200 retrieved Lecture 2 correctly, while 600 retrieved Lecture 4.

Because the quality improvement is more important than the very small latency
difference, I select chunk size 1200.
"""

# DECISION 2 — Retrieval k: 3 vs 5

Now chunk size stays fixed at the current baseline `1200`.

Compare:

```text
k=3 vs k=5
reranker=OFF
```

## What to look at

Ask:

> Do the extra two chunks from k=5 add useful information?

If the first three already contain the needed evidence and chunks 4–5 are weak/noisy, prefer k=3.

If k=5 repeatedly recovers useful evidence missing from k=3, prefer k=5.


In [ ]:
k_3 = run_configuration(
    "k_3",
    {
        "chunk_size": 1200,
        "k": 3,
        "reranker": False,
    },
)

k_5 = run_configuration(
    "k_5",
    {
        "chunk_size": 1200,
        "k": 5,
        "reranker": False,
    },
)

k_comparison = make_side_by_side(
    k_3,
    k_5,
    "k3",
    "k5",
)

display(k_comparison)

display(pd.DataFrame([
    summarize(k_3, "k=3"),
    summarize(k_5, "k=5"),
]))


,case_id,question,k3_top_lecture,k3_lecture_ok,k3_latency,k3_rerank_latency,k3_evidence,k5_top_lecture,k5_lecture_ok,k5_latency,k5_rerank_latency,k5_evidence
0,R1,What is the row picture of a system of linear ...,1,True,0.0445,0.0,"[Let's just check this. If x is one, I have a ...",1,True,0.0161,0.0,"[Let's just check this. If x is one, I have a ..."
1,R2,What is the column picture?,1,True,0.0273,0.0,[this one in the right amounts to get that one...,1,True,0.0194,0.0,[this one in the right amounts to get that one...
2,R3,What is a linear combination in the column pic...,1,True,0.0396,0.0,[this one in the right amounts to get that one...,1,True,0.0210,0.0,[this one in the right amounts to get that one...
3,R4,What is elimination and why do we use it?,2,True,0.0237,0.0,[Okay. This is it. This is then the second lec...,2,True,0.0152,0.0,[Okay. This is it. This is then the second lec...
4,R5,What is discussed about inverse matrices in Le...,4,False,0.0211,0.0,[transposes come in the opposite order. So it'...,4,False,0.0135,0.0,[transposes come in the opposite order. So it'...
5,R6,"In Lecture 4, why does the factorization A = L...",4,True,0.0150,0.0,[is the most basic factorization of a matrix. ...,4,True,0.0224,0.0,[is the most basic factorization of a matrix. ...
6,R7,How is LU related to elimination?,2,False,0.0136,0.0,[It's the matrix that we needed to get this tw...,2,False,0.0161,0.0,[It's the matrix that we needed to get this tw...
7,R8,What is the difference between the row picture...,3,False,0.0143,0.0,[different from taking a row times a column. S...,3,False,0.0135,0.0,[different from taking a row times a column. S...


,configuration,avg_latency_s,correct_top_lecture_rate,avg_reranker_latency_s
0,k=3,0.0249,0.625,0.0
1,k=5,0.0172,0.625,0.0


## Decision point 2


In [ ]:
K_DECISION = 3

K_REASON = """
k=3 and k=5 produced the same correct top-lecture rate: 0.625.

The retrieved evidence shown for the questions is also very similar,
so k=5 did not provide a clear retrieval-quality improvement.

Although k=5 happened to show slightly lower average latency in this run
(0.0172 s vs 0.0249 s), that small difference is not a strong reason
to retrieve two extra chunks.

Because k=3 gives the same observed retrieval quality with less context
passed downstream, I select k=3.
"""

# DECISION 3 — Reranker: OFF vs ON

Keep:

```text
chunk = 1200
k = 3
```

Compare only:

```text
reranker OFF vs ON
```

## What to look at

The reranker is useful if it moves the strongest evidence toward rank #1.

Also compare:

- total latency
- `reranker_latency_s`

The question is:

> Does better ordering justify the extra latency?


In [ ]:
reranker_off = run_configuration(
    "reranker_off",
    {
        "chunk_size": 1200,
        "k": 3,
        "reranker": False,
    },
)

reranker_on = run_configuration(
    "reranker_on",
    {
        "chunk_size": 1200,
        "k": 3,
        "reranker": True,
    },
)

reranker_comparison = make_side_by_side(
    reranker_off,
    reranker_on,
    "off",
    "on",
)

display(reranker_comparison)

display(pd.DataFrame([
    summarize(reranker_off, "reranker OFF"),
    summarize(reranker_on, "reranker ON"),
]))


,case_id,question,off_top_lecture,off_lecture_ok,off_latency,off_rerank_latency,off_evidence,on_top_lecture,on_lecture_ok,on_latency,on_rerank_latency,on_evidence
0,R1,What is the row picture of a system of linear ...,1,True,0.0252,0.0,"[Let's just check this. If x is one, I have a ...",1,True,0.0588,0.0452,"[right, even before the, before the pictures. ..."
1,R2,What is the column picture?,1,True,0.0151,0.0,[this one in the right amounts to get that one...,3,False,0.0450,0.0280,"[here's a, so I call that column one. And what..."
2,R3,What is a linear combination in the column pic...,1,True,0.0135,0.0,[this one in the right amounts to get that one...,1,True,0.0418,0.0263,[this one in the right amounts to get that one...
3,R4,What is elimination and why do we use it?,2,True,0.0148,0.0,[Okay. This is it. This is then the second lec...,2,True,0.0448,0.0255,[Okay. This is it. This is then the second lec...
4,R5,What is discussed about inverse matrices in Le...,4,False,0.0166,0.0,[transposes come in the opposite order. So it'...,2,False,0.0415,0.0275,[So let me make the first step at what's the i...
5,R6,"In Lecture 4, why does the factorization A = L...",4,True,0.0191,0.0,[is the most basic factorization of a matrix. ...,4,True,0.0408,0.0266,[is the most basic factorization of a matrix. ...
6,R7,How is LU related to elimination?,2,False,0.0126,0.0,[It's the matrix that we needed to get this tw...,4,True,0.0415,0.0278,[is the most basic factorization of a matrix. ...
7,R8,What is the difference between the row picture...,3,False,0.0132,0.0,[different from taking a row times a column. S...,3,False,0.0406,0.0266,"[here's a, so I call that column one. And what..."


,configuration,avg_latency_s,correct_top_lecture_rate,avg_reranker_latency_s
0,reranker OFF,0.0163,0.625,0.0000
1,reranker ON,0.0444,0.625,0.0292


## Decision point 3


In [ ]:
RERANKER_DECISION = True

RERANKER_REASON = """
The reranker did not improve the overall correct top-lecture rate in this small benchmark:
both OFF and ON achieved 0.625.

However, reranking changed the ordering of retrieved evidence and improved some individual cases.
For example, R7 changed from an incorrect Lecture 2 top result to the correct Lecture 4 result.

The trade-off is latency:
average latency increased from 0.0163 s to 0.0444 s,
with about 0.0292 s added by reranking.

I keep the reranker ON because reranking changes evidence ordering and adds measurable latency,
while remaining a required architectural component for this project.
"""


# Step 8 — Final decision table

After you fill the three decision cells above, this becomes the final benchmark conclusion.


In [ ]:
final_decision = pd.DataFrame([
    {
        "parameter": "chunk_size",
        "chosen_value": CHUNK_DECISION,
        "reason": CHUNK_REASON,
    },
    {
        "parameter": "retrieval_k",
        "chosen_value": K_DECISION,
        "reason": K_REASON,
    },
    {
        "parameter": "reranker",
        "chosen_value": RERANKER_DECISION,
        "reason": RERANKER_REASON,
    },
])

display(final_decision)


,parameter,chosen_value,reason
0,chunk_size,1200,\n1200-character chunks performed better overa...
1,retrieval_k,3,\nk=3 and k=5 produced the same correct top-le...
2,reranker,True,\nThe reranker did not improve the overall cor...


## Save LangSmith Logs

In [ ]:
runs = list(
    langsmith_client.list_runs(
        project_name="students-channel-brain",
        filter='eq(name, "benchmark_question")',
    )
)

print("Benchmark runs found:", len(runs))

/tmp/ipykernel_1406/529078575.py:2: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  langsmith_client.list_runs(


Benchmark runs found: 64


In [ ]:
langsmith_rows = []

for run in runs:
    langsmith_rows.append({
        "run_id": str(run.id),
        "name": run.name,
        "start_time": run.start_time,
        "end_time": run.end_time,
        "latency_s": (
            (run.end_time - run.start_time).total_seconds()
            if run.end_time and run.start_time
            else None
        ),
        "inputs": run.inputs,
        "outputs": run.outputs,
        "error": run.error,
    })

langsmith_df = pd.DataFrame(langsmith_rows)

display(langsmith_df)

,run_id,name,start_time,end_time,latency_s,inputs,outputs,error
0,01a04ce6-259f-7042-803f-72289ad4ee6e,benchmark_question,2026-08-29 09:42:26.463039+00:00,2026-08-29 09:42:26.473142+00:00,0.010103,"{'case_id': 'R8', 'chunk_size': 600, 'config_n...","{'case_id': 'R8', 'chunk_size': 600, 'config':...",None
1,01a04ce6-2594-70a0-a6b2-8af8149875b6,benchmark_question,2026-08-29 09:42:26.452554+00:00,2026-08-29 09:42:26.462704+00:00,0.010150,"{'case_id': 'R7', 'chunk_size': 600, 'config_n...","{'case_id': 'R7', 'chunk_size': 600, 'config':...",None
2,01a04ce6-258a-7762-b4ea-004a941266b1,benchmark_question,2026-08-29 09:42:26.442136+00:00,2026-08-29 09:42:26.452252+00:00,0.010116,"{'case_id': 'R6', 'chunk_size': 600, 'config_n...","{'case_id': 'R6', 'chunk_size': 600, 'config':...",None
3,01a04ce6-257f-7200-9d57-1d5c3b6a0b1b,benchmark_question,2026-08-29 09:42:26.431763+00:00,2026-08-29 09:42:26.441804+00:00,0.010041,"{'case_id': 'R5', 'chunk_size': 600, 'config_n...","{'case_id': 'R5', 'chunk_size': 600, 'config':...",None
4,01a04ce6-2575-7e42-85c3-678eb698d7c4,benchmark_question,2026-08-29 09:42:26.421210+00:00,2026-08-29 09:42:26.431471+00:00,0.010261,"{'case_id': 'R4', 'chunk_size': 600, 'config_n...","{'case_id': 'R4', 'chunk_size': 600, 'config':...",None
...,...,...,...,...,...,...,...,...
59,01a04ce4-8acd-7600-8939-7ea5bd221d13,benchmark_question,2026-08-29 09:40:41.293710+00:00,2026-08-29 09:40:41.323072+00:00,0.029362,"{'case_id': 'R5', 'chunk_size': 1200, 'config_...","{'case_id': 'R5', 'chunk_size': 1200, 'config'...",None
60,01a04ce4-8aa6-7d82-ac62-682bff9227f4,benchmark_question,2026-08-29 09:40:41.254275+00:00,2026-08-29 09:40:41.293270+00:00,0.038995,"{'case_id': 'R4', 'chunk_size': 1200, 'config_...","{'case_id': 'R4', 'chunk_size': 1200, 'config'...",None
61,01a04ce4-8a91-7a41-8156-a598c49045b0,benchmark_question,2026-08-29 09:40:41.233922+00:00,2026-08-29 09:40:41.253843+00:00,0.019921,"{'case_id': 'R3', 'chunk_size': 1200, 'config_...","{'case_id': 'R3', 'chunk_size': 1200, 'config'...",None
62,01a04ce4-8a79-7871-8ce6-9a8a15276a08,benchmark_question,2026-08-29 09:40:41.209579+00:00,2026-08-29 09:40:41.233456+00:00,0.023877,"{'case_id': 'R2', 'chunk_size': 1200, 'config_...","{'case_id': 'R2', 'chunk_size': 1200, 'config'...",None


In [ ]:
LANGSMITH_EXPORT = (
    OUTPUT_DIR /
    "langsmith_benchmark_question_runs.csv"
)

langsmith_df.to_csv(
    LANGSMITH_EXPORT,
    index=False,
)

print("Saved:", LANGSMITH_EXPORT)

Saved: /content/drive/MyDrive/AI_Engineering_Final_Project/outputs/langsmith_benchmark_question_runs.csv


In [ ]:
APPLICATION_VALIDATION_NOTE = """
Memory and out-of-scope questions were included in the evaluation set,
but they were not executed during the retrieval configuration benchmark.

This benchmark focused on chunk size, retrieval k, and reranker comparison.

Memory and scope protection are application-level behaviors rather than
retrieval-configuration parameters, so they are excluded from the final
configuration score.

They will be validated later as part of the end-to-end Tutor evaluation.
"""

print(APPLICATION_VALIDATION_NOTE)


Memory and out-of-scope questions were included in the evaluation set,
but they were not executed during the retrieval configuration benchmark.

This benchmark focused on chunk size, retrieval k, and reranker comparison.

Memory and scope protection are application-level behaviors rather than
retrieval-configuration parameters, so they are excluded from the final
configuration score.

They will be validated later as part of the end-to-end Tutor evaluation.



In [ ]:
FAILURE_ANALYSIS = [
    {
        "test_id": "R5",
        "question": "What is discussed about inverse matrices in Lecture 3?",
        "expected_lecture": 3,
        "observed_issue": "Top retrieved result came from the wrong lecture.",
        "likely_reason": (
            "The query contains general terms such as 'inverse matrices' "
            "that may also appear in nearby lectures. Semantic retrieval "
            "ranked a related chunk above the requested Lecture 3 chunk."
        ),
        "impact": (
            "The Tutor may receive relevant linear algebra content, "
            "but not the best evidence for the explicitly requested lecture."
        ),
        "possible_future_fix": (
            "When the user explicitly names a lecture, apply a lecture metadata "
            "filter before retrieval."
        ),
    },
    {
        "test_id": "R8",
        "question": "What is the difference between the row picture and the column picture?",
        "expected_lecture": 1,
        "observed_issue": (
            "Retrieval did not consistently rank Lecture 1 as the top result."
        ),
        "likely_reason": (
            "This is a comparison question containing two concepts. "
            "The vector search may retrieve chunks about only one concept "
            "or semantically similar explanations from another lecture."
        ),
        "impact": (
            "The retrieved context may be incomplete for a comparison answer."
        ),
        "possible_future_fix": (
            "Use query decomposition or retrieve evidence separately for "
            "'row picture' and 'column picture' before generating the comparison."
        ),
    },
]

failure_df = pd.DataFrame(FAILURE_ANALYSIS)

display(failure_df)

,test_id,question,expected_lecture,observed_issue,likely_reason,impact,possible_future_fix
0,R5,What is discussed about inverse matrices in Le...,3,Top retrieved result came from the wrong lecture.,The query contains general terms such as 'inve...,The Tutor may receive relevant linear algebra ...,"When the user explicitly names a lecture, appl..."
1,R8,What is the difference between the row picture...,1,Retrieval did not consistently rank Lecture 1 ...,This is a comparison question containing two c...,The retrieved context may be incomplete for a ...,Use query decomposition or retrieve evidence s...


In [ ]:
FAILURE_ANALYSIS_CONCLUSION = """
The selected configuration improved overall retrieval performance,
but it did not solve every retrieval case.

The remaining failures suggest two different weaknesses:

1. Explicit lecture requests may need metadata filtering.
2. Multi-concept comparison questions may need query decomposition
   or retrieval of evidence for each concept separately.

These are system limitations to investigate during the larger
full-course evaluation rather than reasons to continue tuning
chunk size or retrieval k now.
"""

print(FAILURE_ANALYSIS_CONCLUSION)


The selected configuration improved overall retrieval performance,
but it did not solve every retrieval case.

The remaining failures suggest two different weaknesses:

1. Explicit lecture requests may need metadata filtering.
2. Multi-concept comparison questions may need query decomposition
   or retrieval of evidence for each concept separately.

These are system limitations to investigate during the larger
full-course evaluation rather than reasons to continue tuning
chunk size or retrieval k now.



I selected 1200 / k=3 / reranker ON, but failure analysis showed that explicit lecture targeting and multi-concept comparison queries remain challenging. These become candidates for improvement during the later 30+ task evaluation and full-course failure analysis.